## Análisis Exploratorio de Datos (EDA)

### Objetivo
Este notebook realiza un **análisis exploratorio** sobre los datasets extraídos del INE para verificar la integridad, calidad y estructura de los datos antes de las transformaciones.

Se examinan los siete DataFrames generados en la fase de extracción:
- **Empresas constituidas** (`empresas_constituidas.csv`) — Verificación de tipos societarios, consistencia de capital y nº de sociedades.
- **Empresas disueltas** (`empresas_disueltas.csv`) — Distribución por causa de disolución y territorio.
- **IPC** (`ipc.csv`) — Rango de valores, detección de outliers y cardinalidad de dimensiones.
- **Tablas dimensionales** (`sectores_ipc`, `territorio`, `tiempo`, `tipo_medida`) — Validación de claves y completitud.

### Metodología
1. **Carga** de los CSV desde `../files/data_raw/`.
2. **Inspección** mediante `info()`, `describe()`, `sample()` y `value_counts()`.
3. **Detección de inconsistencias** — Identificación de filas duplicadas (como los agregados "Mercantiles" que sumarizan a S.A. y S.L.) y valores anómalos.
4. **Documentación de hallazgos** — Conclusiones que guiarán las transformaciones en la siguiente etapa del pipeline.

In [2]:
# Importación de librerías
import pandas as pd
import numpy as np

# Configuración para visualizar todas las columnas
pd.set_option('display.max_columns', None)

1. Revisamos los archivos exportados para comprobar la integridad de los datos

In [3]:
df_empr_const = pd.read_csv('../files/data_raw/empresas_constituidas.csv')

In [4]:
df_empr_const.sample(5)

,id_const,territorio,id_tiempo,tipo,numero_sociedades,capital
13139,13140,"Asturias, Principado de",201301,S. Comanditarias y S. Colectivas,0,0
10498,10499,Comunitat Valenciana,201302,Sociedades de responsabilidad limitada,1060,85443000
13529,13530,Canarias,201703,S. Comanditarias y S. Colectivas,0,0
9933,9934,Castilla - La Mancha,202307,Sociedades de responsabilidad limitada,241,19143000
439,440,Aragón,200801,Mercantiles,317,35925000


In [5]:
df_empr_const.info()

<class 'pandas.DataFrame'>
RangeIndex: 16720 entries, 0 to 16719
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_const           16720 non-null  int64
 1   territorio         16720 non-null  str  
 2   id_tiempo          16720 non-null  int64
 3   tipo               16720 non-null  str  
 4   numero_sociedades  16720 non-null  int64
 5   capital            16720 non-null  int64
dtypes: int64(4), str(2)
memory usage: 783.9 KB


In [6]:
df_empr_const.describe(include='number').T

,count,mean,std,min,25%,50%,75%,max
id_const,16720.0,8.360500e+03,4.826793e+03,1.0,4180.75,8360.5,12540.25,1.672000e+04
id_tiempo,16720.0,2.016737e+05,5.294115e+02,200801.0,201207.75,201702.5,202109.25,2.026040e+05
numero_sociedades,16720.0,2.134034e+02,4.471220e+02,0.0,0.00,7.0,215.00,3.173000e+03
capital,16720.0,1.544442e+07,1.489657e+08,0.0,0.00,442000.0,7895000.00,1.259172e+10


In [7]:
df_empr_const.describe(include='string').T

,count,unique,top,freq
territorio,16720,19,Andalucía,880
tipo,16720,4,Mercantiles,4180


In [8]:
df_empr_const["territorio"].unique()

<StringArray>
[                  'Andalucía',                      'Aragón',
     'Asturias, Principado de',              'Balears, Illes',
                    'Canarias',                   'Cantabria',
             'Castilla y León',        'Castilla - La Mancha',
                    'Cataluña',        'Comunitat Valenciana',
                 'Extremadura',                     'Galicia',
        'Madrid, Comunidad de',           'Murcia, Región de',
 'Navarra, Comunidad Foral de',                  'País Vasco',
                   'Rioja, La',                       'Ceuta',
                     'Melilla']
Length: 19, dtype: str

Pendiente de arreglar los nombres, minúsculas, tildes, etc. normalizar la columna

In [10]:
df_empr_const["tipo"].unique()

<StringArray>
[                           'Mercantiles',
                    'Sociedades anónimas',
 'Sociedades de responsabilidad limitada',
       'S. Comanditarias y S. Colectivas']
Length: 4, dtype: str

Tras revisar los tipos de empresa, nos damos cuenta que "Mercantiles" es un total de las empresas SL, y SA, vamos a comprobarlo.

In [11]:
df_empr_const[df_empr_const["tipo"] == "Mercantiles"].sample(10)

,id_const,territorio,id_tiempo,tipo,numero_sociedades,capital
259,260,Aragón,202301,Mercantiles,166,5818000
3679,3680,"Rioja, La",201301,Mercantiles,42,2703000
845,846,"Balears, Illes",201011,Mercantiles,154,52667000
1118,1119,Cantabria,202410,Mercantiles,77,1339000
34,35,Andalucía,202306,Mercantiles,1883,62893000
1329,1330,Castilla y León,202507,Mercantiles,255,15551000
562,563,"Asturias, Principado de",201602,Mercantiles,137,4969000
96,97,Andalucía,201804,Mercantiles,1564,165212000
2550,2551,Galicia,201506,Mercantiles,355,12789000
3532,3533,"Rioja, La",202504,Mercantiles,31,397000


In [12]:
df_empr_const["tipo"].value_counts()

tipo
Mercantiles                               4180
Sociedades anónimas                       4180
Sociedades de responsabilidad limitada    4180
S. Comanditarias y S. Colectivas          4180
Name: count, dtype: int64

La razón que los valores totales coinciden es porque hay una fila por fecha, independientemente si hay disueltas o no (0)

In [13]:
df_empr_const[df_empr_const["tipo"] == "Mercantiles"].shape

(4180, 6)

In [14]:
df_empr_const[(df_empr_const["territorio"] == "Melilla") & (df_empr_const["id_tiempo"] == 202505)].head(100)

,id_const,territorio,id_tiempo,tipo,numero_sociedades,capital
3971,3972,Melilla,202505,Mercantiles,10,1037000
8151,8152,Melilla,202505,Sociedades anónimas,0,0
12331,12332,Melilla,202505,Sociedades de responsabilidad limitada,10,1037000
16511,16512,Melilla,202505,S. Comanditarias y S. Colectivas,0,0


In [15]:
df_empr_const[(df_empr_const["territorio"] == "Canarias") & (df_empr_const["id_tiempo"] == 202006)].head(100)

,id_const,territorio,id_tiempo,tipo,numero_sociedades,capital
950,951,Canarias,202006,Mercantiles,186,155088000
5130,5131,Canarias,202006,Sociedades anónimas,0,0
9310,9311,Canarias,202006,Sociedades de responsabilidad limitada,186,155088000
13490,13491,Canarias,202006,S. Comanditarias y S. Colectivas,0,0


In [16]:
df_empr_const[(df_empr_const["territorio"] == "Andalucía") & (df_empr_const["id_tiempo"] == 202407)].head(100)

,id_const,territorio,id_tiempo,tipo,numero_sociedades,capital
21,22,Andalucía,202407,Mercantiles,1564,78298000
4201,4202,Andalucía,202407,Sociedades anónimas,2,1840000
8381,8382,Andalucía,202407,Sociedades de responsabilidad limitada,1562,76458000
12561,12562,Andalucía,202407,S. Comanditarias y S. Colectivas,0,0


Confirmamos nuestras sospechas, procederemos en el paso de transformación a eliminar esas filas. df_empr_const["tipo"] == "Mercantiles"

------------------------------------

In [17]:
df_empr_dis = pd.read_csv('../files/data_raw/empresas_disueltas.csv')

In [18]:
df_empr_dis.sample(5)

,id_dis,territorio,id_tiempo,razon,numero_sociedades
706,707,"Balears, Illes",202206,Voluntaria,60
12525,12526,Melilla,200903,Otras,0
6608,6609,Galicia,202508,Por fusión,8
6140,6141,Cataluña,200908,Por fusión,21
4236,4237,Andalucía,202108,Por fusión,10


In [19]:
df_empr_dis.info()

<class 'pandas.DataFrame'>
RangeIndex: 12540 entries, 0 to 12539
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_dis             12540 non-null  int64
 1   territorio         12540 non-null  str  
 2   id_tiempo          12540 non-null  int64
 3   razon              12540 non-null  str  
 4   numero_sociedades  12540 non-null  int64
dtypes: int64(3), str(2)
memory usage: 490.0 KB


In [20]:
df_empr_dis.shape

(12540, 5)

In [21]:
df_empr_dis.describe(include='number').T

,count,mean,std,min,25%,50%,75%,max
id_dis,12540.0,6270.500000,3620.130523,1.0,3135.75,6270.5,9405.25,12540.0
id_tiempo,12540.0,201673.700000,529.416812,200801.0,201207.75,201702.5,202109.25,202604.0
numero_sociedades,12540.0,32.513238,71.914047,0.0,2.00,8.0,31.00,1114.0


In [22]:
df_empr_dis.describe(include='string').T

,count,unique,top,freq
territorio,12540,19,Andalucía,660
razon,12540,3,Voluntaria,4180


In [23]:
df_empr_dis["razon"].value_counts()

razon
Voluntaria    4180
Por fusión    4180
Otras         4180
Name: count, dtype: int64

La razón que los valores totales coinciden es porque hay una fila por fecha, independientemente si hay disueltas o no (0)

-------------------------

In [24]:
df_ipc = pd.read_csv('../files/data_raw/ipc.csv')

In [25]:
df_ipc.sample(10)

,id_tiempo,id_territorio,id_sector,id_medida,valor_ipc
260294,201703,17,13,1,77.351
175609,201711,12,10,4,-2.200
144531,201907,10,12,2,0.500
36581,200508,4,4,1,76.878
164630,200412,12,1,2,-0.200
185341,201208,13,5,1,82.837
98417,200407,7,14,4,0.800
273408,202302,18,10,2,1.500
114495,200708,8,14,3,3.200
176976,202601,12,12,1,101.591


In [26]:
df_ipc.info()

<class 'pandas.DataFrame'>
RangeIndex: 311752 entries, 0 to 311751
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   id_tiempo      311752 non-null  int64  
 1   id_territorio  311752 non-null  int64  
 2   id_sector      311752 non-null  int64  
 3   id_medida      311752 non-null  int64  
 4   valor_ipc      311752 non-null  float64
dtypes: float64(1), int64(4)
memory usage: 11.9 MB


In [27]:
df_ipc.describe(include='number').T

,count,mean,std,min,25%,50%,75%,max
id_tiempo,311752.0,201377.771331,705.028615,200201.0,200802.00,201403.0,202004.00000,202605.000
id_territorio,311752.0,11.000000,5.477234,2.0,6.00,11.0,16.00000,20.000
id_sector,311752.0,7.500000,4.031135,1.0,4.00,7.5,11.00000,14.000
id_medida,311752.0,2.500000,1.118036,1.0,1.75,2.5,3.25000,4.000
valor_ipc,311752.0,21.504041,37.066418,-22.4,0.10,1.5,37.37975,258.216


In [28]:
df_ipc[df_ipc["valor_ipc"] == -22.4].head()

,id_tiempo,id_territorio,id_sector,id_medida,valor_ipc
169387,202308,12,5,3,-22.4


In [29]:
df_ipc[df_ipc["valor_ipc"] <0].sample(10)

,id_tiempo,id_territorio,id_sector,id_medida,valor_ipc
107118,201112,8,8,2,-0.2
256528,201308,17,9,4,-4.8
16790,201812,3,1,2,-0.6
288965,202010,19,9,3,-3.4
222001,200909,15,8,2,-1.6
273041,200411,18,9,4,-7.7
305776,201108,20,9,4,-2.5
288957,202106,19,9,3,-3.8
157481,201409,11,9,2,-0.1
210695,202401,14,12,4,-0.5


----------------

In [30]:
df_sectores_ipc = pd.read_csv('../files/data_raw/sectores_ipc.csv')

In [31]:
df_sectores_ipc.sample(10)

,id_sector,nombre_sector
12,13,Seguros y servicios financieros
13,14,"Cuidado personal, protección social, y bienes ..."
1,2,Alimentos y bebidas no alcohólicas
7,8,Transporte
0,1,Índice general
9,10,"Actividades recreativas, deporte y cultura"
5,6,"Muebles, artículos del hogar y artículos para ..."
2,3,Bebidas alcohólicas y tabaco
3,4,Vestido y calzado
6,7,Sanidad


In [32]:
df_sectores_ipc.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_sector      14 non-null     int64
 1   nombre_sector  14 non-null     str  
dtypes: int64(1), str(1)
memory usage: 356.0 bytes


In [33]:
df_sectores_ipc.describe(include='number').T

,count,mean,std,min,25%,50%,75%,max
id_sector,14.0,7.5,4.1833,1.0,4.25,7.5,10.75,14.0


In [34]:
df_sectores_ipc.describe(include='string').T

,count,unique,top,freq
nombre_sector,14,14,Índice general,1


In [35]:
df_sectores_ipc["nombre_sector"].unique()

<StringArray>
[                                                                    'Índice general',
                                                 'Alimentos y bebidas no alcohólicas',
                                                       'Bebidas alcohólicas y tabaco',
                                                                  'Vestido y calzado',
                             'Vivienda, agua, electricidad, gas y otros combustibles',
 'Muebles, artículos del hogar y artículos para el mantenimiento corriente del hogar',
                                                                            'Sanidad',
                                                                         'Transporte',
                                                       'Información y comunicaciones',
                                         'Actividades recreativas, deporte y cultura',
                                                                          'Enseñanza',
                             

------------------

In [36]:
df_territorio = pd.read_csv('../files/data_raw/territorio.csv')

In [37]:
df_territorio.sample(10)

,id_territorio,nombre_territorio
1,2,Andalucía
10,11,Comunitat Valenciana
17,18,"Rioja, La"
19,20,Melilla
9,10,Cataluña
15,16,"Navarra, Comunidad Foral de"
18,19,Ceuta
2,3,Aragón
13,14,"Madrid, Comunidad de"
11,12,Extremadura


In [38]:
df_territorio.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_territorio      20 non-null     int64
 1   nombre_territorio  20 non-null     str  
dtypes: int64(1), str(1)
memory usage: 452.0 bytes


In [39]:
df_territorio.describe(include='string').T

,count,unique,top,freq
nombre_territorio,20,20,Nacional,1


In [40]:
df_territorio["nombre_territorio"].unique()

<StringArray>
[                   'Nacional',                   'Andalucía',
                      'Aragón',     'Asturias, Principado de',
              'Balears, Illes',                    'Canarias',
                   'Cantabria',             'Castilla y León',
        'Castilla - La Mancha',                    'Cataluña',
        'Comunitat Valenciana',                 'Extremadura',
                     'Galicia',        'Madrid, Comunidad de',
           'Murcia, Región de', 'Navarra, Comunidad Foral de',
                  'País Vasco',                   'Rioja, La',
                       'Ceuta',                     'Melilla']
Length: 20, dtype: str

-----------------

In [41]:
df_tiempo = pd.read_csv('../files/data_raw/tiempo.csv')

In [42]:
df_tiempo.sample(10)

,id_tiempo,anio,mes,nombre_mes
53,202112,2021,12,Diciembre
227,200706,2007,6,Junio
111,201702,2017,2,Febrero
244,200601,2006,1,Enero
271,200310,2003,10,Octubre
91,201810,2018,10,Octubre
73,202004,2020,4,Abril
267,200402,2004,2,Febrero
175,201110,2011,10,Octubre
208,200901,2009,1,Enero


In [43]:
df_tiempo.info()

<class 'pandas.DataFrame'>
RangeIndex: 294 entries, 0 to 293
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id_tiempo   294 non-null    int64
 1   anio        294 non-null    int64
 2   mes         294 non-null    int64
 3   nombre_mes  294 non-null    str  
dtypes: int64(3), str(1)
memory usage: 9.3 KB


In [44]:
df_tiempo.describe(include='number').T

,count,mean,std,min,25%,50%,75%,max
id_tiempo,294.0,201381.948980,708.657083,200201.0,200802.25,201403.5,202004.75,202606.0
anio,294.0,2013.755102,7.087548,2002.0,2008.00,2014.0,2020.00,2026.0
mes,294.0,6.438776,3.457394,1.0,3.00,6.0,9.00,12.0


---------------------

In [45]:
df_tipo_medida = pd.read_csv('../files/data_raw/tipo_medida.csv')

In [46]:
df_tipo_medida.head(10)

,id_medida,nombre_medida
0,1,Índice
1,2,Variación mensual
2,3,Variación anual
3,4,Variación en lo que va de año


In [47]:
df_tipo_medida.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_medida      4 non-null      int64
 1   nombre_medida  4 non-null      str  
dtypes: int64(1), str(1)
memory usage: 196.0 bytes


In [48]:
df_tipo_medida.value_counts()

id_medida  nombre_medida                
1          Índice                           1
2          Variación mensual                1
3          Variación anual                  1
4          Variación en lo que va de año    1
Name: count, dtype: int64